# Food Calories Prediction Pipeline
This notebook contains the complete pipeline for the food detection and recognition project. It combines all individual scripts into a single sequence.

**Note:** Due to the nature of this project (downloading large datasets, training deep learning models), it is recommended to run this step-by-step.

## Data Downloading
Code from download_dataset.py

In [ ]:
import os
import sys
import json
import time
import shutil
import threading
import subprocess
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────
# CONFIGURATION — all 7 datasets with their slugs and targets
# ─────────────────────────────────────────────────────────────

BASE_DIR = Path("data_folder")   # Change this to your preferred root

DATASETS = [
    # ── Phase 1: Classification datasets (pseudo-bbox) ──────────────────
    {
        "id":          "phase1_food101",
        "phase":       1,
        "slug":        "kmader/food41",
        "target_dir":  BASE_DIR / "datasets/phase1/food101",
        "label":       "Food-101",
        "expected_min_images": 90_000,
        "notes":       "101 global classes, ~101K images",
    },
    {
        "id":          "phase1_chinese",
        "phase":       1,
        "slug":        "yihfeng/chinesefoodnet",
        "target_dir":  BASE_DIR / "datasets/phase1/chinese",
        "label":       "ChineseFoodNet",
        "expected_min_images": 150_000,
        "notes":       "208 Chinese classes, ~185K images",
    },
    {
        "id":          "phase1_mafood",
        "phase":       1,
        "slug":        "theviz/mafood121",
        "target_dir":  BASE_DIR / "datasets/phase1/mafood",
        "label":       "MAFood-121",
        "expected_min_images": 18_000,
        "notes":       "121 multi-cuisine classes, ~21K images",
    },
    # ── Phase 2: Detection datasets (real bounding boxes) ────────────────
    {
        "id":          "phase2_uec256",
        "phase":       2,
        "slug":        "rkuo2000/uecfood256",
        "target_dir":  BASE_DIR / "datasets/phase2/uec256",
        "label":       "UEC-256",
        "expected_min_images": 25_000,
        "notes":       "256 Japanese classes, ~31K images",
    },
    {
        "id":          "phase2_foodrecog22",
        "phase":       2,
        "slug":        "sainikhileshreddy/food-recognition-2022",
        "target_dir":  BASE_DIR / "datasets/phase2/foodrecog2022",
        "label":       "Food Recog 2022",
        "expected_min_images": 38_000,
        "notes":       "498 classes, ~44K COCO-format images",
    },
    {
        "id":          "phase2_foodseg103",
        "phase":       2,
        "slug":        "ggrill/foodseg103",
        "target_dir":  BASE_DIR / "datasets/phase2/foodseg103",
        "label":       "FoodSeg103",
        "expected_min_images": 6_000,
        "notes":       "103 ingredients, ~7K pixel masks",
    },
    {
        "id":          "phase2_unimib",
        "phase":       2,
        "slug":        "dangvanthuc0209/unimib2016",
        "target_dir":  BASE_DIR / "datasets/phase2/unimib",
        "label":       "UNIMIB2016",
        "expected_min_images": 800,
        "notes":       "73 Italian tray classes, ~1K images",
    },
]

# Full project directory tree (created before downloads start)
ALL_DIRS = [
    BASE_DIR / "datasets/phase1/food101",
    BASE_DIR / "datasets/phase1/chinese",
    BASE_DIR / "datasets/phase1/mafood",
    BASE_DIR / "datasets/phase2/uec256",
    BASE_DIR / "datasets/phase2/foodrecog2022",
    BASE_DIR / "datasets/phase2/foodseg103",
    BASE_DIR / "datasets/phase2/unimib",
    BASE_DIR / "merged/phase1/images/train",
    BASE_DIR / "merged/phase1/images/val",
    BASE_DIR / "merged/phase1/labels/train",
    BASE_DIR / "merged/phase1/labels/val",
    BASE_DIR / "merged/phase2/images/train",
    BASE_DIR / "merged/phase2/images/val",
    BASE_DIR / "merged/phase2/labels/train",
    BASE_DIR / "merged/phase2/labels/val",
    BASE_DIR / "runs/phase1",
    BASE_DIR / "runs/phase2",
    BASE_DIR / "weights",
    BASE_DIR / "exports",
    BASE_DIR / "logs",
]

# Thread-safe status store
_lock   = threading.Lock()
_status = {}   # id -> dict with keys: state, start, end, error, image_count


# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────

def log(msg: str, tag: str = "INFO"):
    ts  = datetime.now().strftime("%H:%M:%S")
    tag_str = f"[{tag}]".ljust(9)
    print(f"  {ts}  {tag_str}  {msg}", flush=True)


def set_status(ds_id: str, **kwargs):
    with _lock:
        _status.setdefault(ds_id, {}).update(kwargs)


def count_images(directory: Path) -> int:
    """Count .jpg and .png files recursively."""
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    return sum(1 for p in directory.rglob("*") if p.suffix.lower() in exts)


def check_disk_space(required_gb: float = 150.0):
    total, used, free = shutil.disk_usage(str(BASE_DIR.parent))
    free_gb = free / 1e9
    print(f"\nDisk space check:")
    print(f"    Free:     {free_gb:.1f} GB")
    print(f"    Required: {required_gb:.1f} GB (recommended)")
    if free_gb < required_gb:
        print(f"  ⚠  WARNING: Less than {required_gb} GB free."
              f" Downloads may fail or be incomplete.")
        answer = input("  Continue anyway? [y/N]: ").strip().lower()
        if answer != "y":
            print("  Aborted.")
            sys.exit(1)
    else:
        print(f"    ✓  Sufficient space available.")



def scaffold_directories():
    print("\nCreating project directory tree...")
    created = 0
    for d in ALL_DIRS:
        if not d.exists():
            d.mkdir(parents=True, exist_ok=True)
            created += 1
    total = len(ALL_DIRS)
    print(f"  ✓  {total} directories ready  ({created} newly created)")
    print(f"     Root: {BASE_DIR.resolve()}")


# ─────────────────────────────────────────────────────────────
# DOWNLOAD WORKER  (runs in its own thread per dataset)
# ─────────────────────────────────────────────────────────────

def download_dataset(ds: dict) -> dict:
    """
    Download one dataset using the kaggle CLI, unzip it in-place,
    then count images to verify the download.
    Returns a result dict.
    """
    ds_id      = ds["id"]
    slug       = ds["slug"]
    target     = Path(ds["target_dir"])
    label      = ds["label"]

    set_status(ds_id, state="starting", start=time.time())
    log(f"Starting  {label}  ({ds['notes']})", tag=label[:8])

    try:
        # ── Step 1: Download + unzip via kaggle CLI ──────────────────────
        cmd = [
            sys.executable, "-m", "kaggle",
            "datasets", "download",
            "-d", slug,
            "-p", str(target),
            "--unzip",
        ]

        log(f"Downloading {slug} → {target}", tag=label[:8])
        set_status(ds_id, state="downloading")

        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        output_lines = []
        for line in proc.stdout:
            line = line.rstrip()
            output_lines.append(line)
            # Only print progress-like lines to avoid spam
            if any(kw in line.lower() for kw in
                   ["downloading", "unzip", "extracting", "%", "done", "error", "warning"]):
                log(line, tag=label[:8])

        proc.wait()

        if proc.returncode != 0:
            error_msg = "\n".join(output_lines[-10:])
            raise RuntimeError(
                f"kaggle CLI exited with code {proc.returncode}.\n{error_msg}"
            )

        # ── Step 2: Count images to verify download ──────────────────────
        set_status(ds_id, state="verifying")
        log(f"Verifying image count...", tag=label[:8])

        img_count = count_images(target)
        expected  = ds["expected_min_images"]

        if img_count < expected:
            log(
                f"⚠  Only {img_count:,} images found "
                f"(expected ≥ {expected:,}). Download may be incomplete.",
                tag=label[:8],
            )
            state = "partial"
        else:
            state = "done"

        elapsed = time.time() - _status[ds_id]["start"]
        set_status(ds_id, state=state, end=time.time(), image_count=img_count)

        log(
            f"✓  {label} complete — {img_count:,} images "
            f"in {elapsed/60:.1f} min",
            tag=label[:8],
        )
        return {"id": ds_id, "label": label, "state": state,
                "image_count": img_count, "elapsed_s": elapsed}

    except Exception as e:
        elapsed = time.time() - _status[ds_id].get("start", time.time())
        set_status(ds_id, state="error", end=time.time(),
                   error=str(e), image_count=0)
        log(f"✗  {label} FAILED: {e}", tag="ERROR")
        return {"id": ds_id, "label": label, "state": "error",
                "image_count": 0, "elapsed_s": elapsed, "error": str(e)}


# ─────────────────────────────────────────────────────────────
# SUMMARY PRINTER
# ─────────────────────────────────────────────────────────────

def print_summary(results: list, total_elapsed: float):
    print("\n" + "─" * 60)
    print("  DOWNLOAD SUMMARY")
    print("─" * 60)

    total_images = 0
    errors       = []

    for r in sorted(results, key=lambda x: x["label"]):
        icon  = "✓" if r["state"] == "done" else \
                "~" if r["state"] == "partial" else "✗"
        mins  = r["elapsed_s"] / 60
        imgs  = r.get("image_count", 0)
        total_images += imgs

        line = (
            f"  {icon}  {r['label']:<20s}"
            f"  {imgs:>8,} images"
            f"  {mins:>5.1f} min"
        )
        if r["state"] == "error":
            errors.append(r)
            line += f"  ← ERROR"
        elif r["state"] == "partial":
            line += f"  ← PARTIAL"
        print(line)

    print("─" * 60)
    print(f"     Total images downloaded: {total_images:,}")
    print(f"     Total wall time:         {total_elapsed/60:.1f} min")

    if errors:
        print(f"\nFailed datasets ({len(errors)}):")
        for e in errors:
            print(f"    • {e['label']}: {e.get('error','unknown error')}")
        print("\nRe-run the script — it will skip already-downloaded datasets.")
    else:
        print("\nAll datasets downloaded successfully. ✓")

    # Save log to file
    log_data = {
        "timestamp":      datetime.now().isoformat(),
        "total_images":   total_images,
        "total_elapsed_s": total_elapsed,
        "results":        results,
    }
    log_path = BASE_DIR / "logs" / "download_log.json"
    with open(log_path, "w") as f:
        json.dump(log_data, f, indent=2)
    print(f"\nLog saved → {log_path}")
    print("─" * 60 + "\n")


# ─────────────────────────────────────────────────────────────
# SKIP LOGIC — don't re-download if already complete
# ─────────────────────────────────────────────────────────────

def should_skip(ds: dict) -> bool:
    """Return True if the dataset looks already downloaded."""
    target = Path(ds["target_dir"])
    if not target.exists():
        return False
    img_count = count_images(target)
    if img_count >= ds["expected_min_images"]:
        log(
            f"Skipping {ds['label']} — already has {img_count:,} images",
            tag="SKIP",
        )
        return True
    return False


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def main():
    print("\n" + "═" * 60)
    print("  YOLO26s Food Pipeline — Dataset Downloader")
    print("  Downloads all 7 datasets in parallel")
    print("═" * 60)

    # ── Pre-flight checks ────────────────────────────────────
    check_disk_space(required_gb=150.0)
    scaffold_directories()

    # ── Filter out already-downloaded datasets ───────────────
    pending = [ds for ds in DATASETS if not should_skip(ds)]

    if not pending:
        print("\nAll datasets are already downloaded. Nothing to do.\n")
        return

    print(f"\n Queued for download: {len(pending)} / {len(DATASETS)} datasets")
    for ds in pending:
        print(f"    • {ds['label']:<20s}  {ds['notes']}")

    # ── How many parallel workers? ───────────────────────────
    # Bandwidth is usually the bottleneck, not CPU.
    # 3 parallel downloads is a good balance for most home connections.
    # Change MAX_WORKERS to 1 for a strict sequential download.
    MAX_WORKERS = min(3, len(pending))
    print(f"\n Parallel workers: {MAX_WORKERS}")
    print(f"  Starting at: {datetime.now().strftime('%H:%M:%S')}")
    print("─" * 60)

    wall_start = time.time()
    results    = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_dataset, ds): ds for ds in pending}
        for future in as_completed(futures):
            result = future.result()
            results.append(result)

    # Add skipped datasets to results list for summary
    for ds in DATASETS:
        if ds not in pending:
            img_count = count_images(Path(ds["target_dir"]))
            results.append({
                "id": ds["id"], "label": ds["label"],
                "state": "skipped", "image_count": img_count,
                "elapsed_s": 0,
            })

    total_elapsed = time.time() - wall_start
    print_summary(results, total_elapsed)


if __name__ == "__main__":
    main()

## Data Cleanup - Corrupted Images
Code from corrupted.py

In [ ]:
import glob
from PIL import Image
import json
from tqdm import tqdm

bad = []
all_imgs = glob.glob('data_folder/datasets/**/*.jpg', recursive=True) + \
           glob.glob('data_folder/datasets/**/*.png', recursive=True)

for path in tqdm(all_imgs):
  try:
    img = Image.open(path)
    img.verify()
  except Exception as e:
    bad.append({'path': path, 'error': str(e)})

with open('data_folder/logs/corrupted_images.json','w') as f:
  json.dump(bad, f, indent=2)
print(f'Corrupted: {len(bad)} / {len(all_imgs)}')

## Data Cleanup - Non-RGB Images
Code from non_rgb.py

In [ ]:
import glob
from PIL import Image
import shutil
from tqdm import tqdm

non_rgb = []
all_imgs = glob.glob('data_folder/datasets/**/*.jpg', recursive=True) + \
           glob.glob('data_folder/datasets/**/*.png', recursive=True)
for path in tqdm(all_imgs):
  try:
    img = Image.open(path)
    if img.mode != 'RGB':
      non_rgb.append({'path': path, 'mode': img.mode})
      # Auto-fix: convert to RGB
      img.convert('RGB').save(path)
  except:
    pass

print(f'Non-RGB fixed: {len(non_rgb)}')

## Phase 1 - Rename Directories from CSV
Code from rename_dirs_from_csv.py

In [ ]:
import argparse
import csv
import os
import re
import sys

INVALID_CHARS = r'[<>:"/\\|?*\x00-\x1f]'
TMP_PREFIX = '__tmp_rename__'


def sanitize_name(name: str) -> str:
    name = name.strip()
    name = re.sub(INVALID_CHARS, '_', name)
    name = re.sub(r'[ \t\r\n]+', ' ', name)
    name = name.rstrip(' .')
    return name or '_'


def load_mapping(csv_path: str, id_column: str, name_column: str, pad: int) -> dict:
    mapping = {}
    seen_names = {}
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        if id_column not in reader.fieldnames or name_column not in reader.fieldnames:
            raise ValueError(
                f"CSV must contain columns '{id_column}' and '{name_column}'. Found: {reader.fieldnames}")
        for row in reader:
            raw_id = row[id_column].strip()
            if raw_id == '':
                continue
            try:
                idx = int(float(raw_id))
            except ValueError:
                raise ValueError(f"Invalid numeric id in CSV: '{raw_id}'")
            key = str(idx).zfill(pad)
            label = sanitize_name(row[name_column])
            if not label:
                label = key
            final_name = label
            if final_name in seen_names:
                suffix = seen_names[final_name] + 1
                seen_names[final_name] = suffix
                final_name = f"{label}_{suffix}"
            else:
                seen_names[final_name] = 1
            mapping[key] = final_name
    return mapping


def build_renames(root_dir: str, mapping: dict) -> list:
    dirs = sorted([d for d in os.listdir(root_dir)
                   if os.path.isdir(os.path.join(root_dir, d))])
    renames = []
    for d in dirs:
        if d not in mapping:
            print(f"Warning: directory '{d}' has no entry in the CSV mapping and will be skipped.")
            continue
        new_name = mapping[d]
        if d == new_name:
            continue
        old_path = os.path.join(root_dir, d)
        new_path = os.path.join(root_dir, new_name)
        renames.append((old_path, new_path, d, new_name))
    return renames


def check_conflicts(renames: list, root_dir: str):
    targets = {new for _, new, _, _ in renames}
    if len(targets) != len(renames):
        raise RuntimeError('Duplicate target names found; check the CSV labels.')
    for _, new_path, old_name, new_name in renames:
        if os.path.exists(new_path) and not os.path.isdir(new_path):
            raise RuntimeError(f"Target path exists and is not a directory: {new_path}")
        if os.path.exists(new_path) and os.path.basename(new_path) != os.path.basename(old_name):
            raise RuntimeError(f"Target directory already exists: {new_path}")


def main() -> int:
    parser = argparse.ArgumentParser(description='Rename numbered directories using a CSV label mapping.')
    parser.add_argument('--root', default='data_folder/datasets/phase1/chinese/images',
                        help='Root folder containing numbered directories. Default: %(default)s')
    parser.add_argument('--csv', default='data_folder/datasets/phase1/chinese/food_class.csv',
                        help='CSV file with id/name mapping. Default: %(default)s')
    parser.add_argument('--id-column', default='List No.', help='CSV column for numeric folder IDs. Default: %(default)s')
    parser.add_argument('--name-column', default='English Name', help='CSV column for target folder names. Default: %(default)s')
    parser.add_argument('--pad', type=int, default=3,
                        help='Zero-pad width for numeric folder names (e.g. 000, 001). Default: %(default)s')
    parser.add_argument('--dry-run', action='store_true', help='Show rename operations without executing them.')
    args = parser.parse_args()

    root_dir = os.path.abspath(args.root)
    if not os.path.isdir(root_dir):
        print(f"Error: root folder does not exist: {root_dir}")
        return 1

    mapping = load_mapping(args.csv, args.id_column, args.name_column, args.pad)
    renames = build_renames(root_dir, mapping)
    if not renames:
        print('No directories found to rename.')
        return 0

    print(f"Found {len(renames)} directories to rename in '{root_dir}'.")
    for old_path, new_path, old_name, new_name in renames:
        print(f"{old_name} -> {new_name}")

    if args.dry_run:
        print('\nDry run only; no changes made.')
        return 0

    check_conflicts(renames, root_dir)

    # Use a two-stage rename to avoid collisions with interleaved names
    temp_renames = []
    for old_path, _, old_name, new_name in renames:
        temp_path = os.path.join(root_dir, TMP_PREFIX + os.path.basename(old_path))
        os.rename(old_path, temp_path)
        temp_renames.append((temp_path, os.path.join(root_dir, new_name)))

    for temp_path, final_path in temp_renames:
        os.rename(temp_path, final_path)

    print('\nRename complete.')
    return 0


if __name__ == '__main__':
    raise SystemExit(main())


## Phase 1 - Extract Global Classes
Code from global_class.py

In [ ]:
import os
import pandas as pd
def get_classes(root):
  return sorted([d for d in os.listdir(root)
                 if os.path.isdir(os.path.join(root, d))])
df= pd.read_csv('data_folder/datasets/phase1/chinese/food_class.csv')
chn = df['English Name'].tolist()
f101  = get_classes('data_folder/datasets/phase1/food101/images')
mafd  = get_classes('data_folder/datasets/phase1/mafood/MAFood121/images')

all_classes = sorted(set(
  [c.lower().replace('_',' ') for c in f101 + chn + mafd]
))
print(f'Unique Phase 1 classes: {len(all_classes)}')

with open('data_folder/merged/phase1_classes.txt','w',encoding='utf-8') as f:
  f.write('\n'.join(all_classes))

## Phase 1 - Assign Global IDs
Code from assign_global_id.py

In [ ]:
import json

with open('data_folder/merged/phase1_classes.txt', encoding='utf-8') as f:
  class_list = [l.strip() for l in f]

class_to_id = {name: i for i, name in enumerate(class_list)}
id_to_class = {i: name for name, i in class_to_id.items()}

with open('data_folder/merged/phase1_class_map.json','w',encoding='utf-8') as f:
  json.dump({'class_to_id': class_to_id, 'id_to_class': id_to_class}, f,
            ensure_ascii=False, indent=2)

print(f'Total Phase 1 classes: {len(class_list)}')

## Phase 1 - YOLO Formatting
Code from yolo_formatting.py

In [ ]:
from pathlib import Path
import random
import shutil


def pseudo_label_line(class_id: int) -> str:
  return f'{class_id} 0.5 0.5 1.0 1.0\n'


def show_progress(current: int, total: int, label: str) -> None:
  if total <= 0:
    return
  percent = current / total
  filled = int(30 * percent)
  bar = '█' * filled + '░' * (30 - filled)
  print(f'\r{label}: |{bar}| {current}/{total} ({percent:6.2%})', end='', flush=True)


with open('data_folder/merged/phase1_classes.txt', encoding='utf-8') as f:
  class_list = [l.strip() for l in f]

class_to_id = {name: i for i, name in enumerate(class_list)}
id_to_class = {i: name for name, i in class_to_id.items()}

random.seed(42)
root = Path('data_folder/datasets/phase1/food101/images')
class_folders = [p for p in root.iterdir() if p.is_dir()]
food101_total = sum(len(list(class_folder.glob('*.jpg'))) for class_folder in class_folders)
processed = 0

for class_folder in class_folders:
  class_name = class_folder.name.lower().replace('_',' ')
  class_id   = class_to_id.get(class_name)
  if class_id is None: continue

  images = list(class_folder.glob('*.jpg'))
  random.shuffle(images)
  split = int(len(images)*0.9)
  splits = {'train': images[:split], 'val': images[split:]}

  for subset, imgs in splits.items():
    for src in imgs:
      dst_img = Path(f'data_folder/merged/phase1/images/{subset}/{src.name}')
      dst_lbl = Path(f'data_folder/merged/phase1/labels/{subset}/{src.stem}.txt')
      shutil.copy(src, dst_img)
      dst_lbl.write_text(f'{class_id} 0.5 0.5 1.0 1.0\n')
      processed += 1
      show_progress(processed, food101_total, 'Food-101')

print('\nFood-101 done.')

def convert_classification_dataset(src_root, prefix, subset_ratio=0.9):
  src_root = Path(src_root)
  class_folders = [p for p in src_root.iterdir() if p.is_dir()]
  total_images = 0
  for class_folder in class_folders:
    images = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png'))
    total_images += len(images)

  processed = 0
  for class_folder in class_folders:
    if not class_folder.is_dir(): continue
    class_name = class_folder.name.strip().lower().replace('_',' ')
    class_id   = class_to_id.get(class_name)
    if class_id is None:
      print(f'[WARN] unknown class: {class_name}')
      continue
    images = list(class_folder.glob('*.jpg')) + \
             list(class_folder.glob('*.png'))
    random.shuffle(images)
    split  = int(len(images)*subset_ratio)
    for subset, imgs in [('train',images[:split]),('val',images[split:])]:
      for src in imgs:
        name = f'{prefix}_{src.name}'
        shutil.copy(src, f'data_folder/merged/phase1/images/{subset}/{name}')
        Path(f'data_folder/merged/phase1/labels/{subset}/{Path(name).stem}.txt'
             ).write_text(f'{class_id} 0.5 0.5 1.0 1.0\n')
        processed += 1
        show_progress(processed, total_images, f'{prefix} dataset')

  print()

convert_classification_dataset('data_folder/datasets/phase1/chinese/images', 'chn')
convert_classification_dataset('data_folder/datasets/phase1/mafood/MAFood121/images',  'maf')
print('ChineseFoodNet and MAFood-121 done.')

## Phase 1 - YAML Generation and Validation
Code from yaml_n_validate.py

In [ ]:

from pathlib import Path

import yaml, os

with open('data_folder/merged/phase1_classes.txt', encoding='utf-8') as f:
  class_list = [l.strip() for l in f]

yaml_content = {
  'path':  os.path.abspath('data_folder/merged/phase1'),
  'train': 'images/train',
  'val':   'images/val',
  'nc':    len(class_list),
  'names': class_list
}

with open('data_phase1.yaml', 'w', encoding='utf-8') as f:
  yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

print(f'data_phase1.yaml written — {len(class_list)} classes')

for subset in ['train','val']:
  imgs = {p.stem for p in
          Path(f'data_folder/merged/phase1/images/{subset}').glob('*')}
  lbls = {p.stem for p in
          Path(f'data_folder/merged/phase1/labels/{subset}').glob('*.txt')}
  missing_lbl  = imgs - lbls
  missing_img  = lbls - imgs
  print(f'{subset}: {len(imgs)} images, {len(lbls)} labels')
  print(f'  Missing labels: {len(missing_lbl)}')
  print(f'  Orphan labels:  {len(missing_img)}')

## Phase 1 - Training
Code from phase1_training.py

In [ ]:
import torch
from ultralytics import YOLO
from multiprocessing import freeze_support


def main():

    # --------------------------------------------------
    # Performance Optimizations
    # --------------------------------------------------
    torch.backends.cudnn.benchmark = True

    device = 0 if torch.cuda.is_available() else "cpu"

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU VRAM : {vram_gb:.2f} GB")


    # --------------------------------------------------
    # Data Augmentation
    # --------------------------------------------------

    aug_args = dict(
        mosaic=1.0,
        mixup=0.2,
        degrees=10,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        flipud=0.1,
        hsv_h=0.015,
        hsv_s=0.4,
        hsv_v=0.4,
    )

    # --------------------------------------------------
    # Load Model
    # --------------------------------------------------

    model = YOLO("yolo26s.pt")

    # --------------------------------------------------
    # Training
    # --------------------------------------------------

    model.train(

        data="data_phase1.yaml",

        epochs=50,

        imgsz=640,

        batch=8,

        device=device,

        optimizer="AdamW",

        lr0=1e-3,

        lrf=0.01,

        warmup_epochs=3,

        freeze=10,

        amp=True,

        cache="disk",

        workers=8,

        patience=15,

        save_period=10,

        project="runs/phase1",

        name="pretrain_v1",

        val=False,                 # Validation disabled

        verbose=True,

        **aug_args

    )

    # --------------------------------------------------
    # Validation after training
    # --------------------------------------------------

    best_model = YOLO(
        "runs/detect/runs/phase1/pretrain_v1/weights/best.pt"
    )

    metrics = best_model.val(

        data="data_phase1.yaml",

        batch=2,

        imgsz=640,

        workers=4

    )

    print(metrics)


if __name__ == "__main__":
    freeze_support()
    main()

## Phase 2 - Global Taxonomy
Code from global_taxonomy.py

In [ ]:
import json
from pathlib import Path
from pycocotools.coco import COCO

# =====================================================
# FOOD RECOG 2022
# =====================================================
coco_train = COCO(
    "data_folder/datasets/phase2/foodrecog2022/raw_data/train/annotations.json"
)

fr22_cats = {
    cat["id"]: cat["name"].strip().lower()
    for cat in coco_train.cats.values()
}

print(f"FoodRecog2022 classes : {len(fr22_cats)}")


# =====================================================
# UEC-256
# =====================================================
uec_root = Path("data_folder/datasets/phase2/uec256/UECFOOD256")

category_file = uec_root / "category.txt"

uec_cats = {}

with open(category_file, encoding="utf-8") as f:
    next(f)  # skip header
    for line in f:

        line = line.strip()

        if not line:
            continue

        parts = line.split(maxsplit=1)

        if len(parts) != 2:
            continue

        cat_id = int(parts[0])
        name = parts[1].strip().lower().replace("_", " ")

        uec_cats[cat_id] = name

print(f"UEC-256 classes       : {len(uec_cats)}")


# =====================================================
# FOODSEG103
# =====================================================
foodseg_root = Path(
    "data_folder/datasets/phase2/foodseg103/FoodSeg103"
)

category_file = foodseg_root / "category_id.txt"

foodseg_cats = {}

with open(category_file, encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        parts = line.split(maxsplit=1)

        if len(parts) != 2:
            continue

        cat_id = int(parts[0])
        name = parts[1].strip().lower().replace("_", " ")

        foodseg_cats[cat_id] = name

print(f"FoodSeg103 classes    : {len(foodseg_cats)}")




# =====================================================
# COMBINE ALL NAMES
# =====================================================

all_p2_names = set()

all_p2_names.update(fr22_cats.values())
all_p2_names.update(uec_cats.values())
all_p2_names.update(foodseg_cats.values())


all_p2_names = sorted(all_p2_names)

print("\n--------------------------------")
print(f"Unique Phase-2 classes : {len(all_p2_names)}")
print("--------------------------------")

with open(
    "data_folder/merged/phase2_classes.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(all_p2_names))

print("Saved: data_folder/merged/phase2_classes.txt")

## Phase 2 - Global Taxonomy 2 (Deduplication)
Code from global_taxonomy2.py

In [ ]:
import json
from pathlib import Path
from difflib import get_close_matches

# =====================================================
# Paths
# =====================================================

MERGED = Path("data_folder/merged")
LOGS = Path("data_folder/logs")

LOGS.mkdir(parents=True, exist_ok=True)

PHASE1_CLASSES = MERGED / "phase1_classes.txt"
PHASE2_CLASSES = MERGED / "phase2_classes.txt"

GLOBAL_CLASSES = MERGED / "global_classes.txt"
GLOBAL_MAP = MERGED / "global_class_map.json"
DEDUP_MAP = LOGS / "dedup_map.json"


# =====================================================
# Load Phase 1
# =====================================================

with open(PHASE1_CLASSES, encoding="utf-8") as f:
    phase1_classes = [
        line.strip().lower()
        for line in f
        if line.strip()
    ]

print(f"Phase1 classes : {len(phase1_classes)}")


# =====================================================
# Load Phase 2
# =====================================================

with open(PHASE2_CLASSES, encoding="utf-8") as f:
    phase2_classes = [
        line.strip().lower()
        for line in f
        if line.strip()
    ]

print(f"Raw Phase2 classes : {len(phase2_classes)}")


# =====================================================
# Normalization
# =====================================================

def normalize(name):

    return (
        name.lower()
            .replace("_", " ")
            .replace("-", " ")
            .replace("&", "and")
            .strip()
    )


phase2_classes = [
    normalize(c)
    for c in phase2_classes
]


# =====================================================
# STEP 3
# Fuzzy Deduplicate Phase 2
# =====================================================

print("\nRunning fuzzy matching...")

unique_names = []

dedup_map = {}

manual_review = []

for name in sorted(set(phase2_classes)):

    matches = get_close_matches(
        name,
        unique_names,
        n=1,
        cutoff=0.85
    )

    if matches:

        canonical = matches[0]

        dedup_map[name] = canonical

        # Save possible ambiguous pairs
        score = __import__("difflib").SequenceMatcher(
            None,
            name,
            canonical
        ).ratio()

        if score < 0.92:
            manual_review.append(
                {
                    "name": name,
                    "canonical": canonical,
                    "similarity": round(score, 3)
                }
            )

    else:

        unique_names.append(name)
        dedup_map[name] = name


print(f"Unique Phase2 classes : {len(unique_names)}")


# Save dedup map

with open(
    DEDUP_MAP,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "mapping": dedup_map,
            "manual_review": manual_review
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved {DEDUP_MAP}")


# =====================================================
# STEP 4
# Merge Phase1 + Phase2
# =====================================================

phase1_set = set(phase1_classes)

new_classes = []

for cls in sorted(unique_names):

    if cls not in phase1_set:

        new_classes.append(cls)

global_classes = phase1_classes + new_classes

print(f"New Phase2 classes : {len(new_classes)}")
print(f"Global taxonomy : {len(global_classes)}")


# =====================================================
# Save class list
# =====================================================

with open(
    GLOBAL_CLASSES,
    "w",
    encoding="utf-8"
) as f:

    f.write("\n".join(global_classes))


# =====================================================
# Build maps
# =====================================================

class_to_id = {}

id_to_class = {}

for idx, cls in enumerate(global_classes):

    class_to_id[cls] = idx
    id_to_class[idx] = cls


with open(
    GLOBAL_MAP,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "names": global_classes,
            "map": class_to_id,
            "id_to_name": id_to_class
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved {GLOBAL_MAP}")

print("\nDone.")
print(f"Total global classes : {len(global_classes)}")

## Phase 2 - UEC256 to YOLO
Code from uec256_to_yolo.py

In [ ]:
import shutil

import cv2, os
import json
from pathlib import Path

with open("data_folder/logs/dedup_map.json", encoding="utf-8") as f:
    dedup_map = json.load(f)["mapping"]

with open("data_folder/merged/global_class_map.json", encoding="utf-8") as f:
    global_map = json.load(f)["map"]

def normalize(name):
    return (
        name.lower()
            .replace("_", " ")
            .replace("-", " ")
            .replace("&", "and")
            .strip()
    )

uec_cats = {}
with open("data_folder/datasets/phase2/uec256/UECFOOD256/category.txt", encoding="utf-8") as f:
    next(f)
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            uec_cats[str(parts[0])] = parts[1].strip().lower().replace("_", " ")

def parse_uec256(uec_root):
  uec_root = Path(uec_root)
  # Walk each category folder (named 1, 2, ..., 256)
  for cat_folder in sorted(uec_root.iterdir()):
    if not cat_folder.is_dir(): continue
    cat_id = cat_folder.name   # numeric string '1'..'256'
    bb_file = cat_folder / 'bb_info.txt'
    if not bb_file.exists(): continue
    with open(bb_file) as f:
      lines = f.readlines()[1:]  # skip header
    for line in lines:
      parts = line.strip().split()
      if len(parts) < 5: continue
      img_id, x1, y1, x2, y2 = parts[:5]
      yield cat_id, img_id, int(x1),int(y1),int(x2),int(y2)



for cat_id, img_id, x1,y1,x2,y2 in parse_uec256('data_folder/datasets/phase2/uec256/UECFOOD256'):
  img_path = Path(f'data_folder/datasets/phase2/uec256/UECFOOD256/{cat_id}/{img_id}.jpg')
  if not img_path.exists(): continue
  H, W = cv2.imread(str(img_path)).shape[:2]
  cx = ((x1+x2)/2) / W
  cy = ((y1+y2)/2) / H
  bw = (x2-x1) / W
  bh = (y2-y1) / H
  # Clamp to [0,1] in case of annotation errors
  cx,cy,bw,bh = [max(0,min(1,v)) for v in [cx,cy,bw,bh]]
  
  cat_name = uec_cats.get(str(cat_id))
  if not cat_name: continue
  
  normalized_name = normalize(cat_name)
  canonical_name = dedup_map.get(normalized_name, normalized_name)
  global_id = global_map.get(canonical_name)
  
  if global_id is None: continue

  dst_lbl = Path(f'data_folder/merged/phase2/labels/train/uec_{img_id}.txt')
  with open(dst_lbl, 'a') as f:
    f.write(f'{global_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
  shutil.copy(img_path, f'data_folder/merged/phase2/images/train/uec_{img_id}.jpg')

## Phase 2 - FoodRecog to YOLO
Code from foodrecog_to_yolo.py

In [ ]:
from pycocotools.coco import COCO

coco_train = COCO('data_folder/datasets/phase2/foodrecog2022/raw_data/train/annotations.json')
coco_val   = COCO('data_folder/datasets/phase2/foodrecog2022/raw_data/val/annotations.json')

# Preview annotation structure:
sample_ann = coco_train.loadAnns([1])[0]
print(sample_ann)
# {'id': 1, 'image_id': 123, 'category_id': 42,
#  'bbox': [x, y, w, h], 'segmentation': [...]}
import json
import shutil
from pathlib import Path

with open("data_folder/logs/dedup_map.json", encoding="utf-8") as f:
    dedup_map = json.load(f)["mapping"]

with open("data_folder/merged/global_class_map.json", encoding="utf-8") as f:
    global_map = json.load(f)["map"]

def normalize(name):
    return (
        name.lower()
            .replace("_", " ")
            .replace("-", " ")
            .replace("&", "and")
            .strip()
    )

def convert_coco_to_yolo(coco, img_dir, out_img_dir, out_lbl_dir, prefix):
  for img_id, img_info in coco.imgs.items():
    W, H = img_info['width'], img_info['height']
    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns    = coco.loadAnns(ann_ids)
    if not anns: continue   # skip unannotated images

    src = Path(img_dir) / img_info['file_name']
    name = f'{prefix}_{img_info["file_name"]}'
    dst_img = Path(out_img_dir) / name
    dst_lbl = Path(out_lbl_dir) / (Path(name).stem + '.txt')

    if not src.exists(): continue
    shutil.copy(src, dst_img)

    with open(dst_lbl, 'w') as f:
      for ann in anns:
        x, y, w, h = ann['bbox']
        if w*h == 0: continue   # skip zero-area boxes
        cx = (x + w/2) / W
        cy = (y + h/2) / H
        nw, nh = w/W, h/H
        cat_name = coco.cats[ann['category_id']]['name']
        normalized_name = normalize(cat_name)
        canonical_name = dedup_map.get(normalized_name, normalized_name)
        global_id = global_map.get(canonical_name)
        if global_id is None: continue
        f.write(f'{global_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n')

convert_coco_to_yolo(
  coco_train,
  'data_folder/datasets/phase2/foodrecog2022/raw_data/train/images',
  'data_folder/merged/phase2/images/train',
  'data_folder/merged/phase2/labels/train',
  'fr22'
)

## Phase 2 - FoodSeg to YOLO
Code from foodseg_to_yolo.py

In [ ]:
import json
import cv2
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

# ----------------------------
# Paths
# ----------------------------
ROOT = Path("data_folder/datasets/phase2/foodseg103/FoodSeg103")

IMG_ROOT = ROOT / "Images" / "img_dir"
MASK_ROOT = ROOT / "Images" / "ann_dir"

OUT_ROOT = Path("data_folder/merged/phase2")

TRAIN_IMG_OUT = OUT_ROOT / "images" / "train"
TRAIN_LBL_OUT = OUT_ROOT / "labels" / "train"

VAL_IMG_OUT = OUT_ROOT / "images" / "val"
VAL_LBL_OUT = OUT_ROOT / "labels" / "val"
with open("data_folder/logs/dedup_map.json", encoding="utf-8") as f:
    dedup_map = json.load(f)["mapping"]

with open("data_folder/merged/global_class_map.json", encoding="utf-8") as f:
    global_map = json.load(f)["map"]

def normalize(name):
    return (
        name.lower()
            .replace("_", " ")
            .replace("-", " ")
            .replace("&", "and")
            .strip()
    )

foodseg_cats = {}
with open(ROOT / "category_id.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            foodseg_cats[int(parts[0])] = parts[1].strip().lower().replace("_", " ")

for p in [
    TRAIN_IMG_OUT,
    TRAIN_LBL_OUT,
    VAL_IMG_OUT,
    VAL_LBL_OUT
]:
    p.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Convert one mask to YOLO boxes
# ----------------------------
def mask_to_bboxes(mask_path, img_w, img_h):

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return []

    boxes = []

    classes = np.unique(mask)

    for class_id in classes:

        # background
        if class_id == 0:
            continue

        binary = (mask == class_id).astype(np.uint8)

        n_labels, _, stats, _ = cv2.connectedComponentsWithStats(binary)

        # component 0 is background
        for i in range(1, n_labels):

            x = stats[i, cv2.CC_STAT_LEFT]
            y = stats[i, cv2.CC_STAT_TOP]
            w = stats[i, cv2.CC_STAT_WIDTH]
            h = stats[i, cv2.CC_STAT_HEIGHT]
            area = stats[i, cv2.CC_STAT_AREA]

            # remove tiny noisy regions
            if area < 100:
                continue

            cx = (x + w / 2) / img_w
            cy = (y + h / 2) / img_h

            bw = w / img_w
            bh = h / img_h

            boxes.append(
                (
                    int(class_id),
                    cx,
                    cy,
                    bw,
                    bh,
                )
            )

    return boxes


# ----------------------------
# Process train/test split
# ----------------------------
def process_split(split):

    if split == "train":
        out_img = TRAIN_IMG_OUT
        out_lbl = TRAIN_LBL_OUT
    else:
        out_img = VAL_IMG_OUT
        out_lbl = VAL_LBL_OUT

    img_dir = IMG_ROOT / split
    mask_dir = MASK_ROOT / split

    images = sorted(img_dir.glob("*.jpg"))

    print(f"\nProcessing {split}: {len(images)} images")

    total_boxes = 0

    for img_path in tqdm(images):

        mask_path = mask_dir / (img_path.stem + ".png")

        if not mask_path.exists():
            continue

        image = cv2.imread(str(img_path))

        if image is None:
            continue

        H, W = image.shape[:2]

        boxes = mask_to_bboxes(mask_path, W, H)

        if len(boxes) == 0:
            continue

        new_name = f"seg_{img_path.name}"

        shutil.copy(
            img_path,
            out_img / new_name
        )

        label_path = out_lbl / f"seg_{img_path.stem}.txt"

        with open(label_path, "w") as f:

            for class_id, cx, cy, bw, bh in boxes:

                cat_name = foodseg_cats.get(int(class_id))
                if not cat_name: continue
                
                normalized_name = normalize(cat_name)
                canonical_name = dedup_map.get(normalized_name, normalized_name)
                global_id = global_map.get(canonical_name)
                
                if global_id is None: continue

                f.write(
                    f"{global_id} "
                    f"{cx:.6f} "
                    f"{cy:.6f} "
                    f"{bw:.6f} "
                    f"{bh:.6f}\n"
                )

                total_boxes += 1

    print(f"Finished {split}")
    print(f"Total boxes: {total_boxes}")


# ----------------------------
# Run
# ----------------------------
process_split("train")
process_split("test")

print("\nFoodSeg103 conversion completed.")

## Phase 2 - YAML Generation
Code from yaml_phase2.py

In [ ]:
import json

import yaml, os

with open('data_folder/merged/global_class_map.json', encoding='utf-8') as f:
  gmap = json.load(f)

yaml_content = {
  'path':  os.path.abspath('data_folder/merged/phase2'),
  'train': 'images/train',
  'val':   'images/val',
  'nc':    len(gmap['names']),
  'names': gmap['names']
}
with open('data_phase2.yaml','w',encoding='utf-8') as f:
  yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

print(f'data_phase2.yaml: {len(gmap["names"])} classes')

## Phase 2 - Training
Code from phase2_training.py

In [ ]:
import torch
from ultralytics import YOLO
from multiprocessing import freeze_support


def main():

    torch.backends.cudnn.benchmark = True

    device = 0 if torch.cuda.is_available() else "cpu"

    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU : {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {vram_gb:.2f} GB")
    else:
        print("Running on CPU")

    finetune_args = dict(
        epochs=120,
        imgsz=640,
        batch=8,                   # Change if OOM
        device=device,
        optimizer="AdamW",
        lr0=8e-5,
        lrf=0.001,
        cos_lr=True,
        warmup_epochs=8,
        freeze=10,

        cache=False,
        workers=6,

        amp=True,


        patience=10,
        save_period=5,

        project="runs/phase2",
        name="finetune_v1",
        seed=42,
        deterministic=False,
        multi_scale=False,
        mosaic=0.3,
        close_mosaic=15,
        mixup=0.1,

        degrees=5,
        translate=0.10,
        scale=0.30,

        fliplr=0.5,
        flipud=0.05,

        hsv_h=0.01,
        hsv_s=0.30,
        hsv_v=0.30,
        cls=1.5,
        label_smoothing=0.05,
        val=False,
        verbose=True
    )

    model = YOLO(
        "runs/detect/runs/phase1/pretrain_v1-2/weights/best.pt"
    )

    print("\nStarting Phase-2 Fine-tuning...\n")

    model.train(

        data="data_phase2.yaml",

        **finetune_args

    )

    print("\nLoading best model...\n")

    best_model = YOLO(
        "runs/detect/runs/phase2/finetune_v1/weights/best.pt"
    )

    metrics = best_model.val(

        data="data_phase2.yaml",

        imgsz=640,

        batch=2,

        workers=4

    )

    print("\n==============================")
    print("Final Validation Results")
    print("==============================")

    print(f"mAP@50      : {metrics.box.map50:.4f}")
    print(f"mAP@50-95   : {metrics.box.map:.4f}")
    print(f"Precision   : {metrics.box.mp:.4f}")
    print(f"Recall      : {metrics.box.mr:.4f}")

    print("\nBest model saved at:")
    print("runs/detect/runs/phase2/finetune_v1/weights/best.pt")


if __name__ == "__main__":
    freeze_support()
    main()